In [1]:
import pandas as pd
from pydeseq2.dds import DeseqDataSet

In [2]:
#LOADING

X = pd.read_csv("KIRC_raw_counts.csv", index_col=0)
y = pd.read_csv("KIRC_labels.csv", index_col=0)

In [3]:
metadata = pd.DataFrame({'condition': y.iloc[:, 0]})

In [4]:
print(f"Before filtration: {X.shape[1]} genów.")

Before filtration: 60660 genów.


In [5]:
# FILTERING

genes_to_keep = X.mean(axis=0) >= 1
X_filtered = X.loc[:, genes_to_keep]

In [6]:
print(f"Number of genes after filtration: {X_filtered.shape[1]}")
print(f"Deleted: {X.shape[1] - X_filtered.shape[1]}")

Number of genes after filtration: 35708
Deleted: 24952


In [7]:
# CREATING A DESEQDATASET OBJECT

dds = DeseqDataSet(
    counts=X_filtered,
    metadata=metadata,
    design="~condition",
    n_cpus=4
)

In [8]:
# CALCULATION OF STATISTICAL PARAMETERS (DISPERSION)

dds.deseq2()

Using None as control genes, passed at DeseqDataSet initialization


Fitting size factors...
... done in 2.00 seconds.

Fitting dispersions...
... done in 30.42 seconds.

Fitting dispersion trend curve...
... done in 1.22 seconds.

Fitting MAP dispersions...
... done in 31.54 seconds.

Fitting LFCs...
... done in 18.42 seconds.

Calculating cook's distance...
... done in 2.70 seconds.

Replacing 5741 outlier genes.

Fitting dispersions...
... done in 4.93 seconds.

Fitting MAP dispersions...
... done in 4.61 seconds.

Fitting LFCs...
... done in 3.84 seconds.



In [9]:
# VST TRANSFORMATION

dds.vst()

Fit type used for VST : parametric
Using None as control genes, passed at DeseqDataSet initialization


Fitting size factors...
... done in 1.35 seconds.

Fitting dispersions...
... done in 29.83 seconds.



In [10]:
# SAVING THE NORMALISED DATA TO A NEW TABLE

X_vst = dds.layers['vst_counts']
X_vst = pd.DataFrame(X_vst, index=X_filtered.index, columns=X_filtered.columns)

display(X_vst.iloc[:5, :5])

,ENSG00000000003.15,ENSG00000000005.6,ENSG00000000419.13,ENSG00000000457.14,ENSG00000000460.17
TCGA-CW-5584-01A,11.469653,5.324572,9.966182,8.783311,7.528548
TCGA-CZ-5984-01A,12.422984,4.386262,10.673929,9.063638,7.956131
TCGA-B0-4841-01A,11.346841,7.330588,10.136462,9.185388,7.008919
TCGA-B0-4700-01A,11.372506,4.389160,10.646181,9.074202,7.708191
TCGA-CJ-4905-01A,11.334949,5.026890,10.082451,9.352076,7.732948


In [11]:
# SAVING

X_vst.to_csv("KIRC_vst_normalized.csv")
y.to_csv("KIRC_labels_final.csv")